In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, Descriptors
from rdkit import DataStructs
import numpy as np
import pandas as pd

smiles = "CCO"  # ethanol
mol = Chem.MolFromSmiles(smiles)

print("Molecule loaded:", mol is not None)
print("Number of atoms:", mol.GetNumAtoms())

Molecule loaded: True
Number of atoms: 3


In [2]:
# Morgan fingerprint / circular fingerprint
morgan_fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)

morgan_arr = np.zeros((2048,), dtype=int)
DataStructs.ConvertToNumpyArray(morgan_fp, morgan_arr)

print("Morgan fingerprint length:", len(morgan_arr))
print("Number of active bits:", morgan_arr.sum())
print("First 50 bits:", morgan_arr[:50])

Morgan fingerprint length: 2048
Number of active bits: 6
First 50 bits: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]


[16:07:48] DEPRECATION WARNING: please use MorganGenerator


In [3]:
example_compounds = pd.DataFrame({
    "name": ["ethanol", "aspirin", "caffeine", "ibuprofen"],
    "smiles": [
        "CCO",
        "CC(=O)OC1=CC=CC=C1C(=O)O",
        "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
        "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
    ]
})

example_compounds

,name,smiles
0,ethanol,CCO
1,aspirin,CC(=O)OC1=CC=CC=C1C(=O)O
2,caffeine,Cn1cnc2c1c(=O)n(C)c(=O)n2C
3,ibuprofen,CC(C)CC1=CC=C(C=C1)C(C)C(=O)O


In [4]:
def smiles_to_morgan_array(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return None
    
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    
    return arr

fingerprints = []

for smiles in example_compounds["smiles"]:
    arr = smiles_to_morgan_array(smiles)
    fingerprints.append(arr)

fingerprint_df = pd.DataFrame(fingerprints)
fingerprint_df.insert(0, "name", example_compounds["name"])
fingerprint_df.insert(1, "smiles", example_compounds["smiles"])

fingerprint_df.head()

[16:09:21] DEPRECATION WARNING: please use MorganGenerator
[16:09:21] DEPRECATION WARNING: please use MorganGenerator
[16:09:21] DEPRECATION WARNING: please use MorganGenerator
[16:09:21] DEPRECATION WARNING: please use MorganGenerator


,name,smiles,0,1,2,3,4,5,6,7,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
0,ethanol,CCO,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,aspirin,CC(=O)OC1=CC=CC=C1C(=O)O,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,caffeine,Cn1cnc2c1c(=O)n(C)c(=O)n2C,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,ibuprofen,CC(C)CC1=CC=C(C=C1)C(C)C(=O)O,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
fingerprint_df.to_csv("example_morgan_fingerprints.csv", index=False)
print("Saved example_morgan_fingerprints.csv")

Saved example_morgan_fingerprints.csv


In [1]:
# json

In [2]:
from pathlib import Path
import json

file_path = Path.home() / "predicting-predictability" / "data" / "drug_master.json"

print(file_path)
print("File exists:", file_path.exists())
print("File size MB:", file_path.stat().st_size / 1_000_000)

with open(file_path, "r") as f:
    drug_master = json.load(f)

print(type(drug_master))
print(len(drug_master))

/Users/mell/predicting-predictability/data/drug_master.json
File exists: True
File size MB: 2.959962
<class 'dict'>
6801


In [3]:
if isinstance(drug_master, dict):
    first_key = list(drug_master.keys())[0]
    print("First key:", first_key)
    print(drug_master[first_key])
else:
    print(drug_master[0])

First key: be-2254
{'name': 'be-2254', 'pubchem_cid': '86308637', 'inchi_key': 'PZZOEXPDTYIBPI-MRXNPFEDSA-N', 'smiles': 'Oc1ccc(CCNC[C@H]2CCc3ccccc3C2=O)cc1 |r|', 'broad_id': 'BRD-A24429032-003-03-2', 'moa': 'adrenergic receptor antagonist', 'target': 'ADRA1A', 'clinical_phase': 'Phase 2', 'disease_area': '', 'indication': '', 'fda_approved': False}


In [4]:
import pandas as pd

drugs_df = pd.DataFrame.from_dict(drug_master, orient="index")
drugs_df.insert(0, "record_id", drugs_df.index)

print(drugs_df.shape)
drugs_df.head()

(6801, 12)


,record_id,name,pubchem_cid,inchi_key,smiles,broad_id,moa,target,clinical_phase,disease_area,indication,fda_approved
be-2254,be-2254,be-2254,86308637,PZZOEXPDTYIBPI-MRXNPFEDSA-N,Oc1ccc(CCNC[C@H]2CCc3ccccc3C2=O)cc1 |r|,BRD-A24429032-003-03-2,adrenergic receptor antagonist,ADRA1A,Phase 2,,,False
tasimelteon,tasimelteon,tasimelteon,10220503,PTOIAAWZLUQTIO-GXFFZTMASA-N,CCC(=O)NC[C@@H]1C[C@H]1c1cccc2OCCc12,BRD-K62971431-001-01-9,melatonin receptor agonist,MTNR1A|MTNR1B,Launched,neurology/psychiatry,Non-24-Hour Sleep-Wake Disorder,True
esaxerenone,esaxerenone,esaxerenone,25052023,NOSNHVJANRODGR-UHFFFAOYSA-N,Cc1c(cn(CCO)c1-c1ccccc1C(F)(F)F)C(=O)Nc1ccc(cc...,BRD-K00003374-001-01-9,mineralocorticoid receptor antagonist,,Preclinical,,,False
ixazomib-citrate,ixazomib-citrate,ixazomib-citrate,69040311,YTXSYWAKVMZICI-VBKZILBWSA-N,CC(C)C[C@H](NC(=O)CNC(=O)c1cc(Cl)ccc1Cl)B1OC(=...,BRD-A66419424-001-02-4,proteasome inhibitor,,Launched,hematologic malignancy,multiple myeloma,True
alendronate,alendronate,alendronate,44400013,OGSPWJRAVKPPFI-UHFFFAOYSA-N,NCCCC(O)(P(O)(O)=O)P(O)(O)=O,BRD-K75527158-360-05-6,bone resorption inhibitor,ATP6V1A|FDPS|PTPN4|PTPRE|PTPRS,Launched,orthopedics,osteoporosis,True


In [5]:
smiles_col = "smiles"

drugs_with_smiles = drugs_df[
    drugs_df[smiles_col].notna() &
    (drugs_df[smiles_col].astype(str).str.strip() != "")
].copy()

print("Compounds with SMILES:", drugs_with_smiles.shape[0])
drugs_with_smiles.head()

Compounds with SMILES: 6776


,record_id,name,pubchem_cid,inchi_key,smiles,broad_id,moa,target,clinical_phase,disease_area,indication,fda_approved
be-2254,be-2254,be-2254,86308637,PZZOEXPDTYIBPI-MRXNPFEDSA-N,Oc1ccc(CCNC[C@H]2CCc3ccccc3C2=O)cc1 |r|,BRD-A24429032-003-03-2,adrenergic receptor antagonist,ADRA1A,Phase 2,,,False
tasimelteon,tasimelteon,tasimelteon,10220503,PTOIAAWZLUQTIO-GXFFZTMASA-N,CCC(=O)NC[C@@H]1C[C@H]1c1cccc2OCCc12,BRD-K62971431-001-01-9,melatonin receptor agonist,MTNR1A|MTNR1B,Launched,neurology/psychiatry,Non-24-Hour Sleep-Wake Disorder,True
esaxerenone,esaxerenone,esaxerenone,25052023,NOSNHVJANRODGR-UHFFFAOYSA-N,Cc1c(cn(CCO)c1-c1ccccc1C(F)(F)F)C(=O)Nc1ccc(cc...,BRD-K00003374-001-01-9,mineralocorticoid receptor antagonist,,Preclinical,,,False
ixazomib-citrate,ixazomib-citrate,ixazomib-citrate,69040311,YTXSYWAKVMZICI-VBKZILBWSA-N,CC(C)C[C@H](NC(=O)CNC(=O)c1cc(Cl)ccc1Cl)B1OC(=...,BRD-A66419424-001-02-4,proteasome inhibitor,,Launched,hematologic malignancy,multiple myeloma,True
alendronate,alendronate,alendronate,44400013,OGSPWJRAVKPPFI-UHFFFAOYSA-N,NCCCC(O)(P(O)(O)=O)P(O)(O)=O,BRD-K75527158-360-05-6,bone resorption inhibitor,ATP6V1A|FDPS|PTPN4|PTPRE|PTPRS,Launched,orthopedics,osteoporosis,True


In [6]:
# real morgan

In [7]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
import numpy as np
import pandas as pd

def smiles_to_morgan_array(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    
    if mol is None:
        return None
    
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    
    return arr

fingerprints = []
valid_rows = []
invalid_smiles = []

for idx, row in drugs_with_smiles.iterrows():
    smiles = row["smiles"]
    arr = smiles_to_morgan_array(smiles)
    
    if arr is None:
        invalid_smiles.append((idx, smiles))
    else:
        fingerprints.append(arr)
        valid_rows.append(row)

valid_drugs_df = pd.DataFrame(valid_rows).reset_index(drop=True)

morgan_df = pd.DataFrame(
    fingerprints,
    columns=[f"morgan_{i}" for i in range(2048)]
)

perturbseqr_morgan_df = pd.concat(
    [valid_drugs_df.reset_index(drop=True), morgan_df],
    axis=1
)

print("Valid compounds:", perturbseqr_morgan_df.shape[0])
print("Invalid SMILES:", len(invalid_smiles))
print("Final table shape:", perturbseqr_morgan_df.shape)

perturbseqr_morgan_df.head()

[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerator
[16:45:12] DEPRECATION WARNING: please use MorganGenerat

Valid compounds: 6754
Invalid SMILES: 22
Final table shape: (6754, 2060)


,record_id,name,pubchem_cid,inchi_key,smiles,broad_id,moa,target,clinical_phase,disease_area,...,morgan_2038,morgan_2039,morgan_2040,morgan_2041,morgan_2042,morgan_2043,morgan_2044,morgan_2045,morgan_2046,morgan_2047
0,be-2254,be-2254,86308637,PZZOEXPDTYIBPI-MRXNPFEDSA-N,Oc1ccc(CCNC[C@H]2CCc3ccccc3C2=O)cc1 |r|,BRD-A24429032-003-03-2,adrenergic receptor antagonist,ADRA1A,Phase 2,,...,0,0,0,0,0,0,0,0,0,0
1,tasimelteon,tasimelteon,10220503,PTOIAAWZLUQTIO-GXFFZTMASA-N,CCC(=O)NC[C@@H]1C[C@H]1c1cccc2OCCc12,BRD-K62971431-001-01-9,melatonin receptor agonist,MTNR1A|MTNR1B,Launched,neurology/psychiatry,...,0,0,0,0,0,0,0,0,0,0
2,esaxerenone,esaxerenone,25052023,NOSNHVJANRODGR-UHFFFAOYSA-N,Cc1c(cn(CCO)c1-c1ccccc1C(F)(F)F)C(=O)Nc1ccc(cc...,BRD-K00003374-001-01-9,mineralocorticoid receptor antagonist,,Preclinical,,...,0,0,0,0,0,0,0,0,0,0
3,ixazomib-citrate,ixazomib-citrate,69040311,YTXSYWAKVMZICI-VBKZILBWSA-N,CC(C)C[C@H](NC(=O)CNC(=O)c1cc(Cl)ccc1Cl)B1OC(=...,BRD-A66419424-001-02-4,proteasome inhibitor,,Launched,hematologic malignancy,...,0,0,0,0,0,0,0,0,0,0
4,alendronate,alendronate,44400013,OGSPWJRAVKPPFI-UHFFFAOYSA-N,NCCCC(O)(P(O)(O)=O)P(O)(O)=O,BRD-K75527158-360-05-6,bone resorption inhibitor,ATP6V1A|FDPS|PTPN4|PTPRE|PTPRS,Launched,orthopedics,...,0,0,0,0,0,0,0,0,0,0


In [8]:
from pathlib import Path

output_path = Path.home() / "predicting-predictability" / "data" / "perturbseqr_morgan_fingerprints.csv"

perturbseqr_morgan_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /Users/mell/predicting-predictability/data/perturbseqr_morgan_fingerprints.csv


In [9]:
# Generate MACCS keys for Perturb-Seqr compounds

In [10]:
from rdkit.Chem import MACCSkeys

def smiles_to_maccs_array(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    
    if mol is None:
        return None
    
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((167,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    
    return arr

maccs_fingerprints = []
valid_rows = []
invalid_maccs_smiles = []

for idx, row in drugs_with_smiles.iterrows():
    smiles = row["smiles"]
    arr = smiles_to_maccs_array(smiles)
    
    if arr is None:
        invalid_maccs_smiles.append((idx, smiles))
    else:
        maccs_fingerprints.append(arr)
        valid_rows.append(row)

valid_maccs_drugs_df = pd.DataFrame(valid_rows).reset_index(drop=True)

maccs_df = pd.DataFrame(
    maccs_fingerprints,
    columns=[f"maccs_{i}" for i in range(167)]
)

perturbseqr_maccs_df = pd.concat(
    [valid_maccs_drugs_df.reset_index(drop=True), maccs_df],
    axis=1
)

print("Valid compounds:", perturbseqr_maccs_df.shape[0])
print("Invalid SMILES:", len(invalid_maccs_smiles))
print("Final table shape:", perturbseqr_maccs_df.shape)

perturbseqr_maccs_df.head()

Valid compounds: 6754
Invalid SMILES: 22
Final table shape: (6754, 179)


,record_id,name,pubchem_cid,inchi_key,smiles,broad_id,moa,target,clinical_phase,disease_area,...,maccs_157,maccs_158,maccs_159,maccs_160,maccs_161,maccs_162,maccs_163,maccs_164,maccs_165,maccs_166
0,be-2254,be-2254,86308637,PZZOEXPDTYIBPI-MRXNPFEDSA-N,Oc1ccc(CCNC[C@H]2CCc3ccccc3C2=O)cc1 |r|,BRD-A24429032-003-03-2,adrenergic receptor antagonist,ADRA1A,Phase 2,,...,1,1,1,0,1,1,1,1,1,0
1,tasimelteon,tasimelteon,10220503,PTOIAAWZLUQTIO-GXFFZTMASA-N,CCC(=O)NC[C@@H]1C[C@H]1c1cccc2OCCc12,BRD-K62971431-001-01-9,melatonin receptor agonist,MTNR1A|MTNR1B,Launched,neurology/psychiatry,...,1,1,1,1,1,1,1,1,1,0
2,esaxerenone,esaxerenone,25052023,NOSNHVJANRODGR-UHFFFAOYSA-N,Cc1c(cn(CCO)c1-c1ccccc1C(F)(F)F)C(=O)Nc1ccc(cc...,BRD-K00003374-001-01-9,mineralocorticoid receptor antagonist,,Preclinical,,...,1,1,1,1,1,1,1,1,1,0
3,ixazomib-citrate,ixazomib-citrate,69040311,YTXSYWAKVMZICI-VBKZILBWSA-N,CC(C)C[C@H](NC(=O)CNC(=O)c1cc(Cl)ccc1Cl)B1OC(=...,BRD-A66419424-001-02-4,proteasome inhibitor,,Launched,hematologic malignancy,...,1,1,1,1,1,1,1,1,1,0
4,alendronate,alendronate,44400013,OGSPWJRAVKPPFI-UHFFFAOYSA-N,NCCCC(O)(P(O)(O)=O)P(O)(O)=O,BRD-K75527158-360-05-6,bone resorption inhibitor,ATP6V1A|FDPS|PTPN4|PTPRE|PTPRS,Launched,orthopedics,...,1,1,1,0,1,0,0,1,0,0


In [11]:
output_path = Path.home() / "predicting-predictability" / "data" / "perturbseqr_maccs_keys.csv"

perturbseqr_maccs_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /Users/mell/predicting-predictability/data/perturbseqr_maccs_keys.csv
